# Cypher 기본 문법

# langchain-neo4j 를 이용한 연동

- 설치
    ```bash
    pip install langchain-neo4j
    ```

## `Neo4jGraph`

- `from langchain_neo4j import Neo4jGraph`
- `Neo4jGraph`는 Neo4j Python 드라이버를 감싼 래퍼(Wrapper) 클래스로, Neo4j 데이터베이스와 간편하게 상호작용할 수 있는 인터페이스를 제공한다.

- **주요 역할**
  - **Cypher 쿼리 실행** 
    - 읽기/쓰기 Cypher 쿼리를 실행하고 결과를 `list[dict]`로 반환
  - **그래프 스키마 자동 조회** 
    - 노드 레이블·관계 타입·속성 정보를 자동으로 읽어 `schema` 속성에 저장하여 제공
  - **GraphRAG 파이프라인 연계** 
    - `GraphCypherQAChain`, `LLMGraphTransformer` 등과 결합하여 자연어 기반 그래프 질의응답 구성


###  객체 생성 주요 파라미터

- 구문
    ```python
    Neo4jGraph(
        url=None,
        username=None,
        password=None,
        database=None,
        refresh_schema=True,
        enhanced_schema=False,
    )
    ```

    | 파라미터 | 타입 | 기본값 | 설명 |
    |---|---|---|---|
    | `url` | `str \| None` | `None` | Neo4j 서버 URL (환경변수 `NEO4J_URI` 대체 가능) |
    | `username` | `str \| None` | `None` | 인증 사용자명 (환경변수 `NEO4J_USERNAME` 대체 가능) |
    | `password` | `str \| None` | `None` | 인증 패스워드 (환경변수 `NEO4J_PASSWORD` 대체 가능) |
    | `database` | `str \| None` | `"neo4j"` | 접속할 데이터베이스 이름 |
    | `refresh_schema` | `bool` | `True` | `True`이면 객체 생성 시 자동으로 스키마 정보를 조회 |
    | `enhanced_schema` | `bool` | `False` | `True`이면 DB를 스캔하여 속성의 실제 예시값까지 스키마에 포함 |



- **환경변수에 DB 연결정보가 저장된 경우 객체 생성시 설정하지 않아도 자동으로 읽는다.**
    ```python
    # .env 파일에 설정하면 파라미터 생략 가능
    # NEO4J_URI=bolt://localhost:7687
    # NEO4J_USERNAME=neo4j
    # NEO4J_PASSWORD=password
    # NEO4J_DATABASE=db이름

    graph = Neo4jGraph()  # 환경변수에서 자동 읽기
    ```

### 주요 메소드

- **`query()`** — Cypher 쿼리 실행
  - 조회뿐 아니라 쓰기(CREATE/MERGE) 도 실행
  - 구문
      ```python
        graph.query(query: str, params: dict = {}) -> List[Dict]
      ```
  - Cypher 쿼리를 실행하고 결과를 **딕셔너리 리스트**로 반환
  - `params`로 파라미터화 쿼리 지원 (SQL Injection 방지)

      ```python
      # 읽기 쿼리
      result = graph.query(
          "MATCH (m:Movie) WHERE m.genre = $genre RETURN m.title LIMIT 5",
          params={"genre": "Action"}
      )

      # 쓰기 쿼리 (MERGE / CREATE)    
      graph.query("""
          MERGE (m:Movie {title: 'Top Gun'})
          MERGE (a:Actor {name: 'Tom Cruise'})
          MERGE (a)-[:ACTED_IN]->(m)
      """)
      ```

- **`refresh_schema()`** — 스키마 갱신
  - 구문
    ```python
        graph.refresh_schema()
    ```
  - Neo4j DB로부터 노드 레이블, 관계 타입, 속성 정보를 다시 읽어와 갱신
  - 데이터 구조가 바뀐 후 `schema` 정보를 최신 상태로 유지할 때 사용

- **`add_graph_documents()`** — 그래프 문서 삽입
  - `GraphDocument` 객체에 저장한 `Node`와 `Relationship`들을 Database에 저장한다.

    ```python
        graph.add_graph_documents(
            graph_documents: List[GraphDocument]
        )
    ```

    > - **`GraphDocument`**
    >   - 그래프 구조(노드 + 관계)를 담는 컨테이너 객체
    >   - list[Node] 와 list[Relationship] 으로 구성된다. 
    > - **`Node`**
    >   - Node 정보를 담는 dataclass 로 다음 속성을 가진다.
    >     - `id:str|int`: Node 식별자 
    >     - `type:str`: Node의 Label 
    >     - `properties:dict`: Node 속성
    > - **`Relationship`**
    >   - Relationship 정보를 담는 dataclass 로 다음 속성을 가진다.
    >     - `source: Node`: 시작 노드
    >     - `target: Node`: 끝 노드
    >     - `type: str`: Relationship 타입
    >     - `properties: dict`: 관계 속성

- **`close()`** — 연결 종료
    - 구문
        ```python
        graph.close()
        ```
    - Neo4j 드라이버 연결을 명시적으로 닫습니다. 컨텍스트 매니저(`with` 문)를 사용하면 자동으로 호출됩니다.

###  주요 속성(Instance variable)

| 속성 | 타입 | 설명 |
|---|---|---|
| `schema` | `str` | 현재 DB 스키마 정보 (문자열 형태, LLM 프롬프트에 직접 삽입) |
| `structured_schema` | `dict` | 스키마 정보를 구조화한 딕셔너리 (`node_props`, `rel_props`, `relationships` 키 포함) |

```python
# LLM 프롬프트용 스키마 확인
print(graph.schema)

# 노드 속성 직접 접근
print(graph.structured_schema["node_props"])
```

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

print("NEO4J_URL:", os.getenv("NEO4J_URI"))
print("NEO4J_USERNAME:", os.getenv("NEO4J_USERNAME"))
print("NEO4J_PASSWORD:", os.getenv("NEO4J_PASSWORD"))
print("NEO4J_DATABASE:", os.getenv("NEO4J_DATABASE"))

NEO4J_URL: neo4j://127.0.0.1:7687
NEO4J_USERNAME: neo4j
NEO4J_PASSWORD: asdfghjk
NEO4J_DATABASE: None


In [2]:
from langchain_neo4j import Neo4jGraph
from pprint import pprint
##################################
# 환경변수로 연결설정 정보 저장 후 연결
##################################
graph = Neo4jGraph()

# 연결 확인
print(graph.query("RETURN 'helloworld neo4j' AS result "))

[{'result': 'helloworld neo4j'}]


In [3]:
print(graph.schema)

Node properties:

Relationship properties:

The relationships:



In [4]:
graph.close()

In [3]:
############################
# 직접 값을 넣어 연결
############################
graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
)
print(graph.query("RETURN 'helloworld neo4j' AS result "))

[{'result': 'helloworld neo4j'}]


In [4]:
def reset_database(graph):
    """
    데이터베이스 초기화하기
    """
    # 모든 노드와 관계 삭제
    graph.query("MATCH (n) DETACH DELETE n")
    
    # 모든 제약조건 삭제
    constraints = graph.query("SHOW CONSTRAINTS")
    print(type(constraints)) # list[dict]

    for constraint in constraints:
        constraint_name = constraint.get("name")
        if constraint_name:
            graph.query(f"DROP CONSTRAINT {constraint_name}")
    
    # 모든 인덱스 삭제
    indexes = graph.query("SHOW INDEXES")
    print(type(indexes), indexes)
    for index in indexes:
        index_name = index.get("name")
        index_type = index.get("type")
        if index_name and index_type != "CONSTRAINT":
            graph.query(f"DROP INDEX {index_name}")
    
    print("데이터베이스가 초기화되었습니다.")

# 데이터베이스 초기화
reset_database(graph)

<class 'list'>
<class 'list'> []
데이터베이스가 초기화되었습니다.


# Cypher 쿼리 기초

Cypher는 Neo4j의 그래프 쿼리 언어로, 그래프 패턴을 시각적으로 표현하는 구문을 제공합니다.

## 노드, 속성, 관계 표시
- 노드(Node)는 `()` 소괄호, 관계(Relationship)는 `[]` 대괄호, 속성(Property)는 `{}` 중괄호로 표현
- 라벨은 `:` 콜론으로 시작
- 관계는 반드시 방향(`->` 또는 `<-`)이 있어야 함 (방향 없는 관계는 `CREATE`에서 불가)
- 노드나 관계를 변수에 할당하여 이후에 참조 할 수 있게 한다.
- **예**
   ```cypher
   (n)                     // 모든 노드
   (p:Person)              // Person 레이블의 노드를 조회하고 변수 p에 할당
   (m {name: "Neo4j"})     // name 속성이 "Neo4j"인 노드들을 변수 m에 할당
   (p:Person {age: 30})    // Person 레이블의 age 속성이 30인 노드들을 변수 p에 할당
   -[r:KNOWS]->            // 변수 r에 KNOWS 관계들을 할당
   ```

> ## cypher 주석
>   -  `//`: 한 줄 주석. `//`이 주석의 시작, "엔터"가 주석의 끝.
>   -  `/* */`: 여러 줄 주석. `/*` 이 주석의 시작, `*/`이 주석의 끝.

### **CREATE** 
- 새로운 노드와 관계를 생성한다.
- 동일한 노드가 이미 존재해도 중복 생성되므로 주의해야 한다.
  - 중복 저장을 해결하려면 **constraint**를 설정하거나 **MERGE**를 이용해 저장한다.
- **기본 문법:**

  ```cypher
  // 노드 생성
  CREATE (변수:라벨 {속성키: 속성값, ...})
  ---------------------------------------------------------
  // 관계 생성 (양 노드가 이미 존재할 때)
  MATCH (a), (b)
  WHERE 조건
  CREATE (a)-[:관계타입 {속성}]->(b)
  ---------------------------------------------------------
  // 노드와 관계 한 번에 생성
  CREATE (a:Label1 {...})-[:REL_TYPE]->(b:Label2 {...})
  ```



In [7]:
graph = Neo4jGraph()

# 연결 확인
print(graph.query("RETURN 'helloworld neo4j' AS result "))

[{'result': 'helloworld neo4j'}]


In [5]:
#############################################################
#  Neo4j 데이터베이스에 노드 생성하기 (CREATE 구문 사용)
# CREATE 구문은 새로운 노드나 관계를 생성할 때 사용한다.
#############################################################

cypher_query = """
// (:Person) - Person 타입의 노드 생성
// (:Person {key:value}) - name, age, email 세 개의 property를 가지는 Person 노드 생성 
CREATE (:Person {name: '홍길동', age: 30, email: 'hong@a.com'})
"""

graph.query(cypher_query)

[]

In [6]:
# CREATE 구문을 사용하여 여러 노드를 한 번에 생성할 수 있다.

# 노드 레이블
# - Person
# - City
# Person property: name(str), age(int), gender(str)
# City property: name(str), population(num)

cypher_query = """
CREATE (:Person {name: '김영수', age: 25, gender: '남성'}),
       (:Person {name: '최영희', age: 35, gender: '여성'}),
       (:City {name: '서울', population: '9700000'}),
       (:City {name: '부산', population: '3200000'}),
       (:City {name: '인천', population: '3050000'})
"""

graph.query(cypher_query)

[]

- **관계 표현**: 대시와 대괄호 `-[]->` 사용
   ```cypher
   -->                         // 모든 관계
   -[r:KNOWS]->                // 관계 타입 지정. KNOWS 타입의 관계 
   -[r:LIKES {since: 2020}]->  // 속성을 가진 관계
   <-[r:FOLLOWS]-              // 방향이 반대인 관계
   -[r:FRIENDS|KNOWS]->        // 여러 타입의 관계(OR 조건)
   ```

In [7]:
#########################################
#  기존 노드 사이에 관계 생성하기
########################################
# :Person -[LIVES_IN]-> :City
cypher_query = """
// 기존 노드 조회
MATCH 
    (p:Person {name:'홍길동'}), 
    (c:City {name:'서울'})
CREATE
    (p) -[r:LIVES_IN {since:2019}]-> (c) // MATCH에서 조회한 p, c 노드를 LIVES_IN 관계로 연결
RETURN p, c, r // 세 변수의 값을 호출한 곳으로 반환
"""

graph.query(cypher_query)

[{'p': {'name': '홍길동', 'age': 30, 'email': 'hong@a.com'},
  'c': {'name': '서울', 'population': '9700000'},
  'r': ({'name': '홍길동', 'age': 30, 'email': 'hong@a.com'},
   'LIVES_IN',
   {'name': '서울', 'population': '9700000'})}]

In [8]:
######################################################
#  기존에 생성된 노드들 사이에 관계를 추가하는 Cypher 쿼리
######################################################
# Person -[KNOWS {since:년도}]-> Person
cypher_query = """
MATCH
    (p1: Person {name: "홍길동"}),
    (p2: Person {name: "김영수"})
CREATE (p1)-[r1:KNOWS {since:2024}]->(p2),
       (p2)-[r2:KNOWS {since:2024}]->(p1)
RETURN p1, p2, r1, r2
"""
res = graph.query(cypher_query)

In [9]:
from pprint import pprint
pprint(res)

[{'p1': {'age': 30, 'email': 'hong@a.com', 'name': '홍길동'},
  'p2': {'age': 25, 'gender': '남성', 'name': '김영수'},
  'r1': ({'age': 30, 'email': 'hong@a.com', 'name': '홍길동'},
         'KNOWS',
         {'age': 25, 'gender': '남성', 'name': '김영수'}),
  'r2': ({'age': 25, 'gender': '남성', 'name': '김영수'},
         'KNOWS',
         {'age': 30, 'email': 'hong@a.com', 'name': '홍길동'})}]


In [10]:
########################################
#  노드와 관계를 한 번에 생성하는 Cypher 쿼리
########################################
cypher_query = """
CREATE
    (p1:Person {name:"박명수", age:40, gender:"남성"})-[r:KNOWS]->(p2:Person {name:"유재석", age:40, gender:"남성"})
RETURN p1, p2, r
"""

graph.query(cypher_query)

[{'p1': {'gender': '남성', 'name': '박명수', 'age': 40},
  'p2': {'gender': '남성', 'name': '유재석', 'age': 40},
  'r': ({'gender': '남성', 'name': '박명수', 'age': 40},
   'KNOWS',
   {'gender': '남성', 'name': '유재석', 'age': 40})}]

In [11]:
cypher_query="""
// 손흥민이 LAFDC에서 Play한다.
CREATE
    (p:Person {name:"손흥민", age:35, gender:"남성"})-[r:PLAY_FOR {since:2025}]->(t:Team {name:"LAFC"})
RETURN p AS Player, t AS Team, r AS Relation
"""

result = graph.query(cypher_query)

In [12]:
pprint(result)

[{'Player': {'age': 35, 'gender': '남성', 'name': '손흥민'},
  'Relation': ({'age': 35, 'gender': '남성', 'name': '손흥민'},
               'PLAY_FOR',
               {'name': 'LAFC'}),
  'Team': {'name': 'LAFC'}}]


In [15]:
###########################################################
#  기존에 생성된 노드들 사이에 여러 관계를 추가하는 Cypher 쿼리
###########################################################
cypher_query = """
/*
첫 번째 관계 정보
관계 레이블 : LIVES_IN (거주한다는 의미의 관계 타입)
관계 방향 : a -> c (홍길동이 서울에 산다는 방향성)
관계 속성 : since: 2010 (2010년부터 관계가 시작됨)
관계 생성 의미 : 홍길동은 서울에 2010년부터 거주하기 시작함

두 번째 관계 정보
관계 레이블 : KNOWS (안다/알고 있다는 의미의 관계 타입)
관계 방향 : a -> b (a홍길동이 손흥민을 안다는 방향성)
관계 속성 : since: 2020 (2020년부터 관계가 시작됨)
관계 생성 의미 : 홍길동은 손흥민을 2020년부터 알게 됨

세 번째 관계 정보
관계 레이블 : KNOWS (안다/알고 있다는 의미의 관계 타입)
관계 방향 : b -> a (손흥민이 홍길동을 안다는 방향성)
관계 속성 : since: 2020 (2020년부터 관계가 시작됨)
관계 생성 의미 : 손흥민은 홍길동을 2020년부터 알게 됨
두 번째와 세 번째 관계를 통해 양방향 관계(b <-> a)를 표현함
*/

MATCH
    (p1: Person {name:"홍길동"}), (p2: Person {name:"손흥민"})
CREATE (p1) -[r1:KNOWS {since:2020}]-> (p2),
       (p2) -[r2:KNOWS {since:2020}]-> (p1)
RETURN p1, p2, r1, r2

"""

graph.query(cypher_query)

[{'p1': {'name': '홍길동', 'age': 30, 'email': 'hong@a.com'},
  'p2': {'gender': '남성', 'name': '손흥민', 'age': 35},
  'r1': ({'name': '홍길동', 'age': 30, 'email': 'hong@a.com'},
   'KNOWS',
   {'gender': '남성', 'name': '손흥민', 'age': 35}),
  'r2': ({'gender': '남성', 'name': '손흥민', 'age': 35},
   'KNOWS',
   {'name': '홍길동', 'age': 30, 'email': 'hong@a.com'})}]

In [17]:
######################
# schema 조회
######################
graph.refresh_schema() # 연결 후에 생성된 스키마 정보를 다시 읽어옴
print(graph.schema)


Node properties:
Person {age: INTEGER, email: STRING, name: STRING, gender: STRING}
City {name: STRING, population: STRING}
Team {name: STRING}
Relationship properties:
LIVES_IN {since: INTEGER}
KNOWS {since: INTEGER}
PLAY_FOR {since: INTEGER}
The relationships:
(:Person)-[:LIVES_IN]->(:City)
(:Person)-[:KNOWS]->(:Person)
(:Person)-[:PLAY_FOR]->(:Team)


In [18]:
graph.structured_schema

{'node_props': {'Person': [{'property': 'age', 'type': 'INTEGER'},
   {'property': 'email', 'type': 'STRING'},
   {'property': 'name', 'type': 'STRING'},
   {'property': 'gender', 'type': 'STRING'}],
  'City': [{'property': 'name', 'type': 'STRING'},
   {'property': 'population', 'type': 'STRING'}],
  'Team': [{'property': 'name', 'type': 'STRING'}]},
 'rel_props': {'LIVES_IN': [{'property': 'since', 'type': 'INTEGER'}],
  'KNOWS': [{'property': 'since', 'type': 'INTEGER'}],
  'PLAY_FOR': [{'property': 'since', 'type': 'INTEGER'}]},
 'relationships': [{'end': 'City', 'start': 'Person', 'type': 'LIVES_IN'},
  {'end': 'Person', 'start': 'Person', 'type': 'KNOWS'},
  {'end': 'Team', 'start': 'Person', 'type': 'PLAY_FOR'}],
 'metadata': {'constraint': [], 'index': []}}

### **MATCH** 
- MATCH절은 그래프에서 특정 패턴(노드, 관계, 경로)을 **조회하는** 읽기(read) 명령어이다. SQL의 SELECT에 해당한다.
- `MATCH`만으로는 결과를 보여주지 않으며 항상 `RETURN`(또는 쓰기 절)과 함께 사용
- 노드/관계 자체에 변수를 붙여야 이후에 사용 가능
- 관계 방향은 `->`, `<-` 으로 지정 (`-` 양방향)
   
   ```cypher
   // 기본구문
   MATCH 패턴
   [WHERE 조건1 AND/OR 조건2 ...]
   RETURN ...
   [ORDER BY 정렬기준컬럼 ASC|DESC]

   // 모든 노드 찾기
   MATCH (n) RETURN n;

   // 특정 라벨의 노드 찾기
   MATCH (m:Movie) RETURN m;

   // 속성 조건이 있는 노드 찾기
   MATCH (m:Movie {title: '기생충'}) RETURN m;

   // 관계 패턴 찾기
   MATCH (p:Person)-[r:DIRECTED]->(m:Movie) RETURN p, r, m;

   // 관계 전체를 변수에 저장해서 반환
   MATCH path = (:Person)-[:DIRECTED]->(:Movie) RETURN path
   ```

#### RETURN
- `RETURN` 절은 쿼리의 결과를 어떻게 반환할지 결정한다. 단순한 노드 반환부터 표현식, 별칭, 중복 제거까지 가능하다.
- **기본 문법:**

   ```cypher
   RETURN 변수, 변수.속성, 표현식 [AS 별칭]
   RETURN DISTINCT ...   -- 중복 제거
   ```

- **반환 가능한 것들:**
  - 노드 자체 (`RETURN n`)
  - 속성 (`RETURN n.name`)
  - 관계 (`RETURN r`)
  - 계산 결과 (`RETURN n.born + 10`)
  - 문자열 결합 (`RETURN n.firstName + ' ' + n.lastName`)
  - 컬렉션 (`RETURN [n.name, n.born]`)

In [25]:
###############################
#  모든 노드 조회
###############################
cypher_query = """ 
MATCH (n) RETURN n, labels(n) AS Label // labels(노드 변수): 노드의 타입(Label) 알려줌
"""
res = graph.query(cypher_query)
print(type(res))
print(type(res[0]))
res

<class 'list'>
<class 'dict'>


[{'n': {'name': '홍길동', 'age': 30, 'email': 'hong@a.com'}, 'Label': ['Person']},
 {'n': {'gender': '남성', 'name': '김영수', 'age': 25}, 'Label': ['Person']},
 {'n': {'gender': '여성', 'name': '최영희', 'age': 35}, 'Label': ['Person']},
 {'n': {'name': '서울', 'population': '9700000'}, 'Label': ['City']},
 {'n': {'name': '부산', 'population': '3200000'}, 'Label': ['City']},
 {'n': {'name': '인천', 'population': '3050000'}, 'Label': ['City']},
 {'n': {'gender': '남성', 'name': '박명수', 'age': 40}, 'Label': ['Person']},
 {'n': {'gender': '남성', 'name': '유재석', 'age': 40}, 'Label': ['Person']},
 {'n': {'gender': '남성', 'name': '손흥민', 'age': 35}, 'Label': ['Person']},
 {'n': {'name': 'LAFC'}, 'Label': ['Team']}]

In [26]:
######################################
# 특정 Label의 노드 조회 및 특정 속성 출력
######################################
cypher_query = """
MATCH (n: Person) // Person Label의 노드들만 조회
RETURN n, labels(n) AS Label
"""
graph.query(cypher_query)

[{'n': {'name': '홍길동', 'age': 30, 'email': 'hong@a.com'}, 'Label': ['Person']},
 {'n': {'gender': '남성', 'name': '김영수', 'age': 25}, 'Label': ['Person']},
 {'n': {'gender': '여성', 'name': '최영희', 'age': 35}, 'Label': ['Person']},
 {'n': {'gender': '남성', 'name': '박명수', 'age': 40}, 'Label': ['Person']},
 {'n': {'gender': '남성', 'name': '유재석', 'age': 40}, 'Label': ['Person']},
 {'n': {'gender': '남성', 'name': '손흥민', 'age': 35}, 'Label': ['Person']}]

In [30]:
###############################################################
# 노드와 노드간의 관계를 조회
# (Person 레이블을 가진 노드와 City 레이블을 가진 노드 사이의 관계 조회)
###############################################################
cypher_query = """  
MATCH (p:Person) -[r:LIVES_IN]-> (c:City)
// RETURN p, r, c
RETURN p.name AS Name, c.name AS City, type(r) AS Relationship // type(관계 노드): 관계 노드의 타입 조회
"""

res = graph.query(cypher_query)
pprint(res) #relationship(r) - tuple[src 노드, 관계, target노드]

[{'City': '서울', 'Name': '홍길동', 'Relationship': 'LIVES_IN'}]


In [33]:
# 속성이 일치하는 노드 조회
cypher_query = """
MATCH (p:Person {age:35})  // Person 라벨의 노드 중 age=35인 것 조회
RETURN p
"""
res = graph.query(cypher_query)
res

[{'p': {'gender': '여성', 'name': '최영희', 'age': 35}},
 {'p': {'gender': '남성', 'name': '손흥민', 'age': 35}}]

In [35]:
# 결과가 여러 개일 때 list로 묶어 반환: 내장함수 collect() 사용
cypher_query = """
MATCH (p:Person {age:35})  // Person 라벨의 노드 중 age=35인 것 조회
RETURN collect(p.name) AS Names
"""
res = graph.query(cypher_query)
res

[{'Names': ['최영희', '손흥민']}]

In [38]:
graph.query("MATCH (n) RETURN collect(n.name)")

[{'collect(n.name)': ['홍길동',
   '김영수',
   '최영희',
   '서울',
   '부산',
   '인천',
   '박명수',
   '유재석',
   '손흥민',
   'LAFC']}]

In [40]:
cypher_query = """
MATCH (p:Person {age:35}) 
RETURN DISTINCT p.age AS Age // 중복 제거
"""
res = graph.query(cypher_query)
res

[{'Age': 35}]

#### **WHERE**: 조회 조건 지정
   - `WHERE` 절은 `MATCH` 결과에 추가 조건을 붙여 필터링한다. SQL의 `WHERE`와 거의 동일하다.
   - **기본 문법:**

      ```cypher
      MATCH 패턴
      WHERE 조건1 AND/OR 조건2 ...
      RETURN ...
      ORDER BY 정렬기준컬럼 ASC|DESC
      ```
- **비교/연산자:**

   | 연산자 | 의미 | 사용예시|
   |--------|------|---------|
   | `=`, `<>` | 같다 / 다르다 |WHERE n.age = 30, WHERE n.age <> 30|
   | `<`, `<=`, `>`, `>=` | 범위 비교 |WHERE n.age > 30, WHERE n.age <= 30|
   | `IN [...]` | 리스트 포함 | WHERE n.age IN [20, 30, 40]|
   | `STARTS WITH` | 시작하는 문자열 검색 |WHERE n.name STARTS WITH 'Kim'|
   | `ENDS WITH`| 끝나는 문자열 검색 |WHERE n.name ENDS WITH 'son'|
   | `CONTAINS` | 포함하는 문자열 검색 |n.name CONTAINS 'abc'|
   | `=~` | 정규 표현식식 매칭 |WHERE n.email =~ '.*@gmail.com'|
   | `IS NULL`, `IS NOT NULL` | NULL 체크 | WHERE n.age IS NULL, WHERE n.age IS NOT NULL|
   | `AND`, `OR`, `NOT` | 논리 연산 | WHERE n.age = 30 AND n.name ENDS WITH 'son'|

In [56]:
###############################################################
# 조건 지정 (WHERE 구문)
# WHERE 구문은 MATCH로 찾은 패턴에 추가 조건을 적용할 때 사용
###############################################################
cypher_query = """
MATCH (p:Person)
    // WHERE p.age > 30 // 조건 - property에 대해 조건 비교
    // WHERE p.email IS NOT NULL
    // WHERE p.email IS NULL
    // WHERE p.name STARTS WITH '홍'
    WHERE p.age IN [30, 40]
RETURN p
"""

graph.query(cypher_query)

[{'p': {'name': '홍길동', 'age': 30, 'email': 'hong@a.com'}},
 {'p': {'gender': '남성', 'name': '박명수', 'age': 40}},
 {'p': {'gender': '남성', 'name': '유재석', 'age': 40}}]

In [60]:
cypher_query = """
MATCH (p: Person) -[r:LIVES_IN]-> (c: City)
// WHERE p.age >= 30 AND c.name = '부산' // '서울'
WHERE p.age >= 30 OR c.name = '부산'
RETURN p, c
"""

graph.query(cypher_query)

[{'p': {'name': '홍길동', 'age': 30, 'email': 'hong@a.com'},
  'c': {'name': '서울', 'population': '9700000'}}]

#### ORDER BY / SKIP / LIMIT
- 쿼리 결과를 정렬하고, 일부만 가져오고, 건너뛴다.
- **기본 문법:**

    ```cypher
    RETURN ...
    ORDER BY 컬럼1 [ASC|DESC], 컬럼2 [ASC|DESC]
    SKIP N        -- 앞에서 N개 건너뛰기
    LIMIT M;      -- 최대 M개만 반환
    ```

- `ORDER BY`의 기본은 오름차순(`ASC`). 내림차순은 `DESC`.
- `SKIP`과 `LIMIT`은 페이징(pagination)에 자주 사용된다.

In [63]:
#################################################
#  나이 순 정렬 - ORDER BY와 LIMIT 사용 예제
#################################################
cypher_query = """
MATCH (p: Person)
RETURN p
ORDER BY p.age // ASC(기본), DESC
LIMIT 3 // 앞의 3개만 반환
"""

result = graph.query(cypher_query)
result

[{'p': {'gender': '남성', 'name': '김영수', 'age': 25}},
 {'p': {'name': '홍길동', 'age': 30, 'email': 'hong@a.com'}},
 {'p': {'gender': '여성', 'name': '최영희', 'age': 35}}]

In [67]:
#######################################
#  SKIP - 조회결과에서 앞 n개를 건너 뛴다.
#######################################
cypher_query = """
MATCH (p: Person)
RETURN p
ORDER BY p.age, p.name
SKIP 2 // 앞 두 개는 건너뛰고 반환
LIMIT 2
"""

result = graph.query(cypher_query)
result

[{'p': {'gender': '남성', 'name': '손흥민', 'age': 35}},
 {'p': {'gender': '여성', 'name': '최영희', 'age': 35}}]

### OPTIONAL MATCH
- `MATCH`는 관계 패턴이 일치하지 않으면 결과 행 자체가 조회되지 않는다. 반면 `OPTIONAL MATCH`는 일치하지 않아도 행을 유지하고 매칭되지 않은 변수는 `null`이 된다. SQL의 `LEFT OUTER JOIN`과 유사하다.
- **기본 문법:**

    ```cypher
    MATCH (a:Label)
    OPTIONAL MATCH (a)-[:REL]->(b)
    RETURN a, b
    ```
- master 정보를 match로 찾고 부가 정보를 optional match로 정의한다.

In [70]:
#############################
# OPTIONAL MATCH
#############################
cypher_query = """
// MATCH (p: Person) -[r: PLAY_FOR]-> (t: Team)

// 모든 Person 노드들이 다 조회되고 Team과 PLAY_FOR 관계가 있는 Person은 Team 정보도 같이 나오게 함
MATCH (p: Person) // 메인 정보
WHERE p.age > 30
OPTIONAL MATCH (p) -[:PLAY_FOR]-> (t: Team) // Team 부가 정보
RETURN p, t
"""

result = graph.query(cypher_query)
result

[{'p': {'gender': '여성', 'name': '최영희', 'age': 35}, 't': None},
 {'p': {'gender': '남성', 'name': '박명수', 'age': 40}, 't': None},
 {'p': {'gender': '남성', 'name': '유재석', 'age': 40}, 't': None},
 {'p': {'gender': '남성', 'name': '손흥민', 'age': 35}, 't': {'name': 'LAFC'}}]

In [ ]:
cypher_query = """
// MATCH(c: City) -[LIVES_IN]-> (p:Person)
// 모든 도시는 다 조회. 도시에 살고 있는 사람이 있으면 추가 정보로 조회
MATCH (c: City)
OPTIONAL MATCH (p: Person) -[:LIVES_IN]-> (c)
RETURN c, p
"""

result = graph.query(cypher_query)
for r in result:
    print(r)
    print("----------------------------")

{'c': {'name': '서울', 'population': '9700000'}, 'p': {'name': '홍길동', 'age': 30, 'email': 'hong@a.com'}}
----------------------------
{'c': {'name': '부산', 'population': '3200000'}, 'p': None}
----------------------------
{'c': {'name': '인천', 'population': '3050000'}, 'p': None}
----------------------------


### **MERGE**

- `MERGE`는 "있으면 찾고(MATCH), 없으면 만든다(CREATE)"
- `MERGE`는 **명시한 모든 속성**을 매칭 키로 사용한다. 그래서 매칭 키 외 속성은 `ON CREATE SET`에서 추가하는 것이 안전하다.
  - `MERGE (p:Person {name:'홍길동', age: 30})` 은 name이 홍길동이고 age가 30인 Person 이 있으면 조회 없으면 생성한다.
- **기본 문법:**

    ```cypher
    // 노드 MERGE
    MERGE (n:Label {key: value})

    // 관계 MERGE (보통 양 노드를 먼저 MATCH/MERGE한 뒤)
    MATCH (a), (b) WHERE ...
    MERGE (a)-[:REL]->(b)

    // MERGE에 따른 추가 동작
    // ON CREATE SET: 생성했을 때 실행되어 속성을 추가한다.
    // ON MERGE SET: 조회했을 때 실행되어 속성을 추가/변경한다.
    MERGE (n:Label {id: 1})
    ON CREATE SET n.createdAt = timestamp(), n.count = 1
    ON MATCH SET n.updatedAt = timestamp(), n.count = n.count + 1
    ```

In [74]:
############################################################
# MERGE: 조회 또는 생성
# 이름이 손흥민인 사람이 있으면 조회하고 없으면 생성한다.
############################################################
cypher_query = """
// MERGE (p: Person {name: "손흥민"})
MERGE (p: Person {name: "김민재"})
RETURN p
"""

result = graph.query(cypher_query)
for d in result:
    print(d)
    print("---------------------------"*3)

{'p': {'name': '김민재'}}
---------------------------------------------------------------------------------


In [ ]:
##################################################################################
# 이름이 박지성인 사람이 있으면 조회하고 없으면 생성한다.
## 박지성이 생성되었으면 p.address = '수원' 속성을 추가한다.
## 박지성이 조회되었으면 p.address = '맨체스터' 속성을 추가 또는 변경한다.
##################################################################################
cypher_query = """
MERGE (p :Person {name: "박지성"})
ON CREATE SET p.address = "수원", p.age = 45 // 생성할 때 속성 변경
ON MATCH SET p.address = "맨체스터"
RETURN p
"""

result = graph.query(cypher_query)
for d in result:
    print(d)
    print("---------------------------"*3)

{'p': {'address': '수원', 'name': '김지성', 'age': 45}}
---------------------------------------------------------------------------------


In [78]:
##########################################################################
# 관계가 있으면 조회하고 없으면 추가한다.
## 김영수와 서울 사이에 LIVES_IN 관계가 있으면 조회하고 없으면 생성한다.
##########################################################################
cypher_query = """
MATCH (p: Person {name: "김영수"}), (c: City {name: "부산"})
MERGE (p) -[:LIVES_IN]-> (c)
RETURN p, c
"""

result = graph.query(cypher_query)
for d in result:
    print(d)
    print("---------------------------"*3)

{'p': {'gender': '남성', 'name': '김영수', 'age': 25}, 'c': {'name': '부산', 'population': '3200000'}}
---------------------------------------------------------------------------------


### **SET** 
- SET 절은 이미 존재하는 노드나 관계의 **속성(property), 라벨(label)을 추가하거나 수정**할 때 사용한다.
- CREATE가 새 노드나 관계를 만드는 절이라면, SET은 조회된 데이터의 값을 변경하는 절이다.

- **기본 문법:**

   ```cypher
   MATCH 패턴
   SET n.속성 = 값                // 단일 속성 변경/추가
   SET n.속성1 = v1, n.속성2 = v2 // 속성 변경/추가 (한개 속성=값)
   SET n += {key: value, ...}     // 속성 변경/추가 (map객체를 이용해서 한번에 여러 개 병합)
   SET n = {key: value, ...}      // 전체 교체(주의: 기존 속성 삭제됨)
   SET n:NewLabel                 // 라벨 추가
   ```

In [80]:
#######################################
#  노드 속성 업데이트 (SET 구문 사용) -> 1. 바꿀 대상 조회 2. 변경/추가
#######################################
cypher_query = """
MATCH (p: Person {name: "홍길동"})
SET p.age = 35, p.hobby = "게임"  // age는 변경, hobby는 추가
RETURN p
"""

graph.query(cypher_query)

[{'p': {'name': '홍길동', 'age': 35, 'email': 'hong@a.com', 'hobby': '게임'}}]

In [ ]:
cypher_query = """
MATCH (p: Person {name: "손흥민"})
SET p += {age: 36, email: "son@mail.com"} // age 변경, email 추가
RETURN p
"""

graph.query(cypher_query)

[]

In [ ]:
cypher_query = """
MATCH (p: Person {name: "김지성"})
SET p = {name: "김지성", emai: "son@mail.com"} // property 자체 변경
RETURN p
"""

graph.query(cypher_query)
# [{'p': {'address': '수원', 'name': '김지성', 'age': 45, 'emai': 'son@mail.com'}}]

[{'p': {'name': '김지성', 'emai': 'son@mail.com'}}]

In [91]:
cypher_query = """
// 레이블이 없는 노드 추가
CREATE (p {name: "이강인", age: 21})
RETURN p
"""

graph.query(cypher_query)

[{'p': {'name': '이강인', 'age': 21}}]

In [93]:
############################
#  레이블 추가 및 속성 설정 
############################
cypher_query = """
MATCH (p {name: "이강인"})
SET p:Person // Label - :Label 이름
RETURN p
"""

graph.query(cypher_query)

[{'p': {'name': '이강인', 'age': 21}}]

In [95]:
cypher_query = """
MATCH (p1 {name: "이강인"}), (p2 {name: "손흥민"})
SET p1: Player, p2: Player
RETURN p1, p2
"""

graph.query(cypher_query)

[{'p1': {'name': '이강인', 'age': 21},
  'p2': {'gender': '남성', 'name': '손흥민', 'age': 36, 'emai': 'son@mail.com'}}]

### **DELETE**, **DETACH DELETE**
- `DELETE`: 노드/관계 삭제
  - Neo4j는 관계가 연결된 노드를 `DELETE`로 삭제할 수 없다.
- `DETACH DELETE`: 노드와 그에 연결된 모든 관계를 함께 삭제
- **기본 문법:**

   ```cypher
      // 관계만 삭제
      MATCH (a)-[r:REL]->(b) DELETE r;

      // 노드만 삭제 (연결 관계가 없을 때)
      MATCH (n:Label) DELETE n;

      // 노드 + 연결된 모든 관계 삭제
      MATCH (n:Label) DETACH DELETE n;

      // 전체 그래프 초기화
      MATCH (n) DETACH DELETE n;

   ```

In [101]:
###################
#  노드 삭제
###################
cypher_query = """
MATCH (p: Person|Player {name: "이강인"}) // p: Person|Player # Person 또는 Player 타입
DELETE p
"""

graph.query(cypher_query)

[]

In [103]:
#################################################
#  관계가 있는 노드 삭제 (DETACH DELETE 사용)
#################################################
cypher_query = """
// MATCH (p: Person {name: "유재석"}) // relationship 있으면 삭제 불가
// DELETE p
// MATCH (p: Person {name: "유재석"})-[r]-() // 유재석과 모든 관계의 모든 노드 
// DELETE r // 관계 삭제

MATCH (p: Person {name: "유재석"})
DETACH DELETE p // p와 연결된 관계 모두 삭제한 후에 p 삭제
"""

graph.query(cypher_query)

[]

In [105]:
################
# 관계를 삭제
################
cypher_query = """
MATCH (:Person {name: "홍길동"}) -[r]- (:Person {name: "손흥민"})
DELETE r
"""
graph.query(cypher_query)

[]

### **REMOVE**
- 레이블이나 속성 제거
   ```cypher
   
      // 속성 삭제
      MATCH (n) REMOVE n.속성명;

      // 노드의 라벨 삭제
      MATCH (n) REMOVE n:라벨명;
   ```

In [ ]:
############################################ 
# 속성(Property) 제거 (REMOVE 구문 사용)
############################################  
cypher_query = """
MATCH (p: Person {name: "홍길동"})
REMOVE p.hobby, p.age // 노드의 property
RETURN p
"""
graph.query(cypher_query)

In [106]:
######################################
# 레이블(Label) 제거 (REMOVE 구문 사용)
######################################
cypher_query = """
MATCH (p: Person {name: "홍길동"})
REMOVE p: Person
RETURN p
"""

graph.query(cypher_query)

[{'p': {'name': '홍길동', 'age': 35, 'email': 'hong@a.com', 'hobby': '게임'}}]